# 🚀 나만의 비트코인 딥러닝 트레이딩 모델 (Advanced Strategy)

**과제 목표 달성을 위한 고도화 전략**
1. **데이터 업그레이드**: 볼린저 밴드, 스토캐스틱 등 보조지표 추가로 시장 상황 판단력 강화
2. **모델 업그레이드**: 금융 시계열에 효율적인 GRU 모델 적용 및 최적화
3. **전략 고도화**: 확률 기반 진입 + 손절매(Stop-Loss) 로직으로 안정적인 수익 추구

---

In [ ]:
# 1. 환경 설정 및 라이브러리 임포트
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# 기존 utils.py에서 필요한 함수 불러오기
from utils import (
    load_bitcoin_data,
    evaluate_model,
    plot_confusion_matrix,
    device,
    calculate_rsi,
    calculate_macd
)

# 시각화 설정
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print(f"✅ 설정 완료! Using device: {device}")

## 2. 데이터 로딩 및 피처 엔지니어링 강화 (Feature Engineering)

기존 피처(이동평균, RSI, MACD)에 더해 **볼린저 밴드**와 **스토캐스틱**을 추가하여 시장의 과매수/과매도 구간을 더 정밀하게 포착합니다.

In [ ]:
def create_advanced_features(df, lookback_days=10):
    """
    기존 피처에 보조지표를 추가하여 강화된 데이터셋 생성
    """
    data = df.copy()
    
    # MultiIndex 처리
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    
    close = data['Close'].squeeze()
    high = data['High'].squeeze()
    low = data['Low'].squeeze()
    volume = data['Volume'].squeeze()
    
    # --- 기존 피처 ---
    data['Returns'] = close.pct_change()
    
    # 이동평균
    for window in [5, 10, 20, 60]:  # 60일선 추가 (중기 추세)
        ma = close.rolling(window=window).mean()
        data[f'MA_{window}'] = ma
        data[f'MA_{window}_ratio'] = close / ma
        
    # 변동성
    returns = data['Returns']
    for window in [5, 20]:
        data[f'Volatility_{window}'] = returns.rolling(window=window).std()
        
    # RSI & MACD
    data['RSI_14'] = calculate_rsi(close, period=14)
    macd, macd_signal = calculate_macd(close)
    data['MACD'] = macd
    data['MACD_Signal'] = macd_signal
    data['MACD_Hist'] = macd - macd_signal  # 히스토그램 추가
    
    # --- [NEW] 추가 보조지표 ---
    
    # 1. 볼린저 밴드 (Bollinger Bands)
    # 가격이 밴드 상단을 돌파하면 과매수, 하단을 이탈하면 과매도
    bb_window = 20
    bb_std = 2
    bb_ma = close.rolling(window=bb_window).mean()
    bb_sigma = close.rolling(window=bb_window).std()
    
    data['BB_Upper'] = bb_ma + (bb_std * bb_sigma)
    data['BB_Lower'] = bb_ma - (bb_std * bb_sigma)
    # 밴드폭 (변동성 지표)
    data['BB_Width'] = (data['BB_Upper'] - data['BB_Lower']) / bb_ma
    # 현재 가격의 밴드 내 위치 (0~1)
    data['BB_Position'] = (close - data['BB_Lower']) / (data['BB_Upper'] - data['BB_Lower'])
    
    # 2. 스토캐스틱 (Stochastic Oscillator)
    # 최근 N일간의 최고가/최저가 대비 현재가 위치
    stoch_window = 14
    stoch_low = low.rolling(window=stoch_window).min()
    stoch_high = high.rolling(window=stoch_window).max()
    
    # Fast %K
    data['Stoch_K'] = 100 * ((close - stoch_low) / (stoch_high - stoch_low))
    # Slow %D (Fast %K의 3일 이동평균)
    data['Stoch_D'] = data['Stoch_K'].rolling(window=3).mean()
    
    # 3. 모멘텀 (Momentum)
    data['Momentum'] = close - close.shift(10)
    
    # 과거 수익률 (Lag Features)
    for lag in range(1, lookback_days + 1):
        data[f'Returns_Lag_{lag}'] = returns.shift(lag)
        
    # 타겟 변수: 다음 날 가격 상승(1) / 하락(0)
    data['Target'] = (close.shift(-1) > close).astype(int)
    
    return data.dropna()

# 데이터 로드 및 피처 생성
start_date = "2018-01-01"  # 데이터 기간 확대 (더 많은 학습 데이터)
end_date = datetime.now().strftime("%Y-%m-%d")

raw_data = load_bitcoin_data(start_date=start_date, end_date=end_date)
btc_features = create_advanced_features(raw_data)

print(f"\n데이터 shape: {btc_features.shape}")
print(f"사용된 피처 수: {len(btc_features.columns) - 7}")  # Target 등 제외

In [ ]:
# 데이터 전처리 (스케일링 및 시퀀스 생성)
from utils import prepare_data

# 1. 데이터 분할
X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(
    btc_features, test_size=0.15, validation_size=0.15  # 테스트 비중 조정
)

# 2. 스케일링
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# 3. 시퀀스 생성
sequence_length = 30  # 30일치 데이터를 보고 내일 예측

def create_sequences(X, y, seq_len=30):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len])
    return np.array(X_seq), np.array(y_seq)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train.values, sequence_length)
X_val_seq, y_val_seq = create_sequences(X_val_scaled, y_val.values, sequence_length)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test.values, sequence_length)

# 4. DataLoader
batch_size = 64  # 배치 사이즈 증가
train_loader = DataLoader(TensorDataset(torch.FloatTensor(X_train_seq), torch.FloatTensor(y_train_seq)), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.FloatTensor(X_val_seq), torch.FloatTensor(y_val_seq)), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.FloatTensor(X_test_seq), torch.FloatTensor(y_test_seq)), batch_size=batch_size, shuffle=False)

print(f"학습 데이터: {X_train_seq.shape}")

## 3. GRU 모델 구현 (Model Upgrade)

LSTM보다 구조가 간단하여 학습 속도가 빠르고, 금융 시계열 데이터에서 종종 더 나은 성능을 보이는 **GRU(Gated Recurrent Unit)** 모델을 사용합니다.

In [ ]:
class AdvancedGRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.3):
        super(AdvancedGRUModel, self).__init__()
        
        # GRU Layer 1
        self.gru1 = nn.GRU(input_size, hidden_size, num_layers=1, 
                          batch_first=True, dropout=0)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.dropout1 = nn.Dropout(dropout)
        
        # GRU Layer 2
        self.gru2 = nn.GRU(hidden_size, hidden_size//2, num_layers=1, 
                          batch_first=True, dropout=0)
        self.bn2 = nn.BatchNorm1d(hidden_size//2)
        self.dropout2 = nn.Dropout(dropout)
        
        # Fully Connected Layers
        self.fc1 = nn.Linear(hidden_size//2, 32)
        self.relu = nn.ReLU()
        self.dropout3 = nn.Dropout(dropout)
        
        self.fc2 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        # x: (batch, seq_len, input_size)
        
        out, _ = self.gru1(x)
        out = self.dropout1(out)
        # BatchNorm 적용을 위해 차원 변경: (batch, hidden, seq) -> (batch, seq, hidden)
        out = out.permute(0, 2, 1)
        out = self.bn1(out)
        out = out.permute(0, 2, 1)
        
        out, _ = self.gru2(out)
        # 마지막 시퀀스만 사용
        out = out[:, -1, :]
        out = self.dropout2(out)
        out = self.bn2(out)
        
        out = self.fc1(out)
        out = self.relu(out)
        out = self.dropout3(out)
        
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

# 모델 초기화
model = AdvancedGRUModel(
    input_size=X_train_seq.shape[2],
    hidden_size=128,
    dropout=0.3
).to(device)

print(model)

In [ ]:
# 모델 학습 함수 (Early Stopping 포함)
def train_model_advanced(model, train_loader, val_loader, epochs=100, lr=0.0005, patience=20):
    criterion = nn.BCELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)  # AdamW 사용
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    print("🚀 학습 시작...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch.unsqueeze(1))
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            train_correct += (predicted.squeeze() == y_batch).sum().item()
            train_total += y_batch.size(0)
            
        avg_train_loss = train_loss / len(train_loader)
        train_acc = train_correct / train_total
        
        # 검증
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch.unsqueeze(1))
                
                val_loss += loss.item()
                predicted = (outputs > 0.5).float()
                val_correct += (predicted.squeeze() == y_batch).sum().item()
                val_total += y_batch.size(0)
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total
        
        # Learning Rate 스케줄링
        scheduler.step(avg_val_loss)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}')
            
        # Early Stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"🛑 Early stopping at epoch {epoch+1}")
                break
                
    if best_model_state:
        model.load_state_dict(best_model_state)
        
    return history

# 학습 실행
history = train_model_advanced(model, train_loader, val_loader, epochs=150, lr=0.001, patience=20)

In [ ]:
# 학습 결과 시각화
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.title('Accuracy Curve')
plt.legend()
plt.show()

## 4. 트레이딩 전략 고도화 (Strategy Upgrade)

단순 확률 비례 투자가 아닌, **안전 장치(Stop-Loss)**와 **확신 진입(Threshold)**을 결합한 전략입니다.

1.  **진입 조건**: 상승 확률이 `buy_threshold` (예: 0.6) 이상일 때만 매수
2.  **청산 조건**:
    *   하락 확률이 높을 때 (`prob < 0.5`)
    *   **손절매 (Stop-Loss)**: 매수 가격 대비 `stop_loss_pct` (예: -3%) 이상 하락 시 즉시 매도
3.  **포지션 사이징**: 확률에 따라 투자 비중 조절 (Kelly Criterion 개념 일부 차용)

In [ ]:
def simulate_advanced_strategy(model, X_test, y_test, prices, dates, 
                             initial_capital=10000, fee=0.001, 
                             buy_threshold=0.6, stop_loss_pct=0.03):
    
    model.eval()
    with torch.no_grad():
        probs = model(torch.FloatTensor(X_test).to(device)).cpu().numpy().flatten()
    
    cash = initial_capital
    btc_amount = 0
    portfolio_values = []
    buy_price = 0  # 손절매 기준용
    
    trade_log = []
    
    for i in range(len(probs)):
        current_price = prices[i]
        prob = probs[i]
        date = dates[i]
        
        # 현재 가치 평가
        current_val = cash + (btc_amount * current_price)
        portfolio_values.append(current_val)
        
        # 마지막 날 전량 매도
        if i == len(probs) - 1:
            if btc_amount > 0:
                cash += btc_amount * current_price * (1 - fee)
                btc_amount = 0
            continue
            
        # --- 전략 로직 ---
        
        # 1. 손절매 체크 (보유 중일 때만)
        if btc_amount > 0:
            loss_pct = (current_price - buy_price) / buy_price
            if loss_pct < -stop_loss_pct:
                # 손절매 발동!
                sell_val = btc_amount * current_price * (1 - fee)
                cash += sell_val
                btc_amount = 0
                trade_log.append(f"{date}: 🛑 Stop Loss Triggered! Sold at {current_price:.2f} (Loss: {loss_pct:.2%})")
                continue  # 이번 턴 종료

        # 2. 매수 (현금 보유 시)
        if btc_amount == 0 and prob >= buy_threshold:
            # 확신하는 만큼 투자 (Kelly Criterion 단순화: 확률 비례)
            # 너무 소액은 거래 안 함
            invest_amt = cash * prob  # 확률만큼 투자
            
            buy_amount = (invest_amt * (1 - fee)) / current_price
            btc_amount += buy_amount
            cash -= invest_amt
            buy_price = current_price
            trade_log.append(f"{date}: 🟢 Buy at {current_price:.2f} (Prob: {prob:.2f})")
            
        # 3. 매도 (보유 시)
        elif btc_amount > 0 and prob < 0.5:
            # 하락 예측 시 매도
            sell_val = btc_amount * current_price * (1 - fee)
            cash += sell_val
            btc_amount = 0
            trade_log.append(f"{date}: 🔴 Sell at {current_price:.2f} (Prob: {prob:.2f})")
            
    # 최종 결과 계산
    final_return = (portfolio_values[-1] - initial_capital) / initial_capital * 100
    
    return {
        'final_value': portfolio_values[-1],
        'return': final_return,
        'portfolio_values': portfolio_values,
        'trade_log': trade_log
    }

# 테스트 데이터 준비
test_start_idx = len(btc_features) - len(y_test) + sequence_length
test_prices = btc_features['Close'].iloc[test_start_idx:test_start_idx+len(y_test_seq)].values
test_dates = btc_features.index[test_start_idx:test_start_idx+len(y_test_seq)]

# 전략 실행
result = simulate_advanced_strategy(
    model, X_test_seq, y_test_seq, test_prices, test_dates,
    buy_threshold=0.6,  # 60% 이상 확신 시 매수
    stop_loss_pct=0.03  # 3% 손실 시 손절
)

print(f"💰 최종 자본: ${result['final_value']:,.2f}")
print(f"📈 수익률: {result['return']:.2f}%")
print(f"📝 거래 횟수: {len([l for l in result['trade_log'] if 'Buy' in l])}회 매수")

In [ ]:
# 5. 벤치마크 비교 (Buy and Hold)
buy_hold_return = (test_prices[-1] - test_prices[0]) / test_prices[0] * 100
buy_hold_values = [10000 * (p / test_prices[0]) for p in test_prices]

plt.figure(figsize=(15, 7))
plt.plot(test_dates, buy_hold_values, label=f'Buy & Hold ({buy_hold_return:.2f}%)', linestyle='--', color='gray')
plt.plot(test_dates, result['portfolio_values'], label=f'My AI Strategy ({result["return"]:.2f}%)', color='red', linewidth=2)
plt.title('Performance Comparison: My AI vs Buy & Hold')
plt.legend()
plt.show()